# Large-amplitude vibration of a cantilever — HHT

A minimal comparison of a **secant-stiffness iteration** and a **Newton-type tangent iteration**, using only LowLevelFEM.jl. The beam is modeled as a 2D plane-stress continuum with finite rotations and Saint-Venant–Kirchhoff (Green–Lagrange) strain.

In [1]:
using LowLevelFEM

structured_rect_mesh(lx=1.0, ly=0.04, y0=-0.02, n=2, order=2)
mat = Material("body"; E=1e6, ν=0.3, ρ=1000.0)
U = Field([mat]; type=:VectorField, dim=2, fieldName=:u)
b, D2 = 0.01, D(:PlaneStress, mat)
support = [BoundaryCondition("left"; field=U, ux=0.0, uy=0.0)]
u0 = initialDisplacement(U, "body"; ux=0.0, uy=0.0)
v0 = initialVelocity(U, "body"; vx=0.0, vy=(x,y,z)->0.25*x^2*(3-x)/2)
M = b*mat.ρ*∫(U⋅U)
K = b*∫(SymGrad(U)⋅D2⋅SymGrad(U))
B0 = [1 0 0 0; 0 0 0 1; 0 1 1 0];

## 1. Secant-stiffness iteration

With $F=I+\nabla_Xu$ and $E=(F^TF-I)/2$, the internal force is represented by $f_{\mathrm{int}}(u)=K_s(u)u$. The HHT solver updates $K_s$ at each nonlinear iteration.

In [2]:
function deformation(state)
    H = ∇(expandTo3D(state.fields[1]))
    F = tensorToMatrix(unitTensor(H) + H)
    B = [F[1,1] 0 F[2,1] 0;
         0 F[1,2] 0 F[2,2];
         F[1,2] F[1,1] F[2,2] F[2,1]]
    return H, B
end

function secant(A, state)
    H, B = deformation(state)
    b*∫(Grad(U)⋅B'⋅D2⋅((B+B0)/2)⋅Grad(U); gauss=4)
end

secant (generic function with 1 method)

In [3]:
@time u_sec, v_sec = HHT(K, M, 0*u0, u0, v0, 401, 0.02;
    support, α=-0.05, modifyA0=secant,
    nonlinearoptions=(maxiter=50, abstol=1e-5),
    copyOperators=false);

 58.932907 seconds (176.19 M allocations: 25.759 GiB, 10.15% gc time, 33.50% compilation time)


## 2. Newton-type tangent iteration

The analytical Total Lagrangian tangent contains a material and a geometric part:

$$K_t=\int_{\Omega_0}G^T\bigl(B^TDB+G_\sigma\bigr)G\,b\,d\Omega_0.$$

To keep the *same internal force* as in the secant calculation, the right-hand-side callback applies

$$b_{\mathrm{eq}}(u)=f_{\mathrm{ext}}+K_t(u)u-K_s(u)u.$$

**Discretization note:** the stress used in $G_\sigma$ is constructed as a LowLevelFEM field and interpolated to quadrature points. Thus this is an **analytical tangent / Newton-type iteration**, not a verified exact Jacobian of the discrete secant residual. An exactly consistent Gauss-point tangent requires evaluating stress from the deformation gradient at the same quadrature points (as in a dedicated element formulation).

In [4]:
function tangent(A, state)
    H, B = deformation(state)
    h11, h12, h21, h22 = H[1,1], H[1,2], H[2,1], H[2,2]
    e11 = h11 + (h11*h11 + h21*h21)/2
    e22 = h22 + (h12*h12 + h22*h22)/2
    γ12 = h12 + h21 + h11*h12 + h21*h22
    s11 = D2[1,1]*e11 + D2[1,2]*e22
    s22 = D2[2,1]*e11 + D2[2,2]*e22
    s12 = D2[3,3]*γ12
    Gσ = [s11 s12 0 0; s12 s22 0 0;
          0 0 s11 s12; 0 0 s12 s22]
    b*∫(Grad(U)⋅B'⋅D2⋅B⋅Grad(U) +
        Grad(U)⋅Gσ⋅Grad(U); gauss=4)
end

function equivalent_rhs(rhs, state)
    u = vec(state.fields[1].a)
    rhs.a[:,1] .+= state.operators.A0.A*u - secant(nothing, state).A*u
    return rhs
end

equivalent_rhs (generic function with 1 method)

In [5]:
@time u_new, v_new = HHT(K, M, 0*u0, u0, v0, 401, 0.02;
    support, α=-0.05, modifyA0=tangent, modifyRHS=equivalent_rhs,
    nonlinearoptions=(maxiter=50, abstol=1e-5),
    copyOperators=false);

 55.305244 seconds (180.28 M allocations: 22.748 GiB, 9.25% gc time, 6.45% compilation time)


## 3. Compare the responses

The two calculations use the same mass, HHT parameters, boundary conditions, initial conditions and internal-force residual. Compare the displacement histories; a smaller nonlinear iteration count alone does not establish greater accuracy.

In [6]:
println("Maximum DOF difference: ", maximum(abs.(u_sec.a - u_new.a)))
showDoFResults(u_sec; name="Cantilever — secant HHT", factor=1.0, visible=true)
showDoFResults(u_new; name="Cantilever — Newton-type HHT", factor=1.0, visible=true)
openPostProcessor()

Maximum DOF difference: 3.405072510898055e-7


XOpenIM() failed
Fontconfig warning: using without calling FcInit()
